# 4.6 — The UDF Performance Hierarchy, Measured

**Chapter 4, section 4.4**, and the starting point for **Exercises 5 and 6**.

**The question this notebook answers:** the chapter arranges four ways of applying a function to
a column into a hierarchy, ordered by how data crosses the boundary between the JVM and Python:

| Tier | Form | Transport |
|------|------|-----------|
| 1 | built-in column functions | no crossing at all |
| 2 | `@arrow_udf` | columnar batches, consumed as Arrow arrays |
| 3 | `@pandas_udf` | columnar batches, converted to pandas objects |
| 4 | `@udf` | one row at a time |

The chapter says to search from the top and stop at the first tier that can express the
transformation. This notebook implements **one** transformation in all four tiers, on two
million taxi fares, and times them — so the hierarchy is measured rather than asserted.

It also does two things the chapter's figure cannot: it reads the default of
`spark.sql.execution.pythonUDF.arrow.enabled` on this version, which changes what tier 4
actually costs; and it reproduces the batch-local trap that makes a vectorized UDF quietly
wrong.

Runs on a laptop in about three minutes.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile, time, logging
import numpy as np
import pandas as pd
import pyarrow as pa
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import udf, pandas_udf, arrow_udf
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-4.6")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

SHIPPED_DEFAULT = spark.conf.get("spark.sql.execution.pythonUDF.arrow.enabled")
# Turned off deliberately, and the reason is the subject of a section below: on this version
# the Arrow-optimized row-at-a-time UDF path stalls in local mode at these row counts.
spark.conf.set("spark.sql.execution.pythonUDF.arrow.enabled", "false")
for _lg in ("SQLQueryContextLogger", "DataFrameQueryContextLogger"):
    logging.getLogger(_lg).setLevel(logging.CRITICAL)

print("Spark", spark.version, "| PyArrow", pa.__version__, "| pandas", pd.__version__)
print("arrow batch size (maxRecordsPerBatch):",
      spark.conf.get("spark.sql.execution.arrow.maxRecordsPerBatch"))
print("arrow batch size (maxBytesPerBatch)  :",
      spark.conf.get("spark.sql.execution.arrow.maxBytesPerBatch"))
print("Arrow optimization for plain @udf    :", SHIPPED_DEFAULT,
      "as shipped, now set to",
      spark.conf.get("spark.sql.execution.pythonUDF.arrow.enabled"), "-- see the last section")

Spark 4.2.0 | PyArrow 25.0.1 | pandas 2.3.3
arrow batch size (maxRecordsPerBatch): 10000
arrow batch size (maxBytesPerBatch)  : 67108864b
Arrow optimization for plain @udf    : true as shipped, now set to false -- see the last section


## The input, materialized once

Both sides of a benchmark must start from the same place. Two columns are read out of the
compressed CSV, tripled to six million rows, and written to Parquet in scratch. Every timed run
below then reads that same Parquet file from scratch: no run is handed a cache the others were
denied, and the scan cost is identical for all of them, so the differences are the functions.

In [2]:
names = ["medallion", "hack_license", "pickup_datetime", "dropoff_datetime",
         "trip_time", "trip_distance", "pickup_longitude", "pickup_latitude",
         "dropoff_longitude", "dropoff_latitude", "payment_type", "fare_amount",
         "surcharge", "mta_tax", "tip_amount", "tolls_amount", "total_amount"]
types = {"medallion": StringType(), "hack_license": StringType(),
         "pickup_datetime": TimestampType(), "dropoff_datetime": TimestampType(),
         "trip_time": IntegerType(), "payment_type": StringType()}
schema = StructType([StructField(n, types.get(n, DoubleType()), True) for n in names])

source = (spark.read.schema(schema).option("header", "false")
          .csv(f"{DATA}/taxi-data-sorted-small.csv.bz2")
          .select("fare_amount", "medallion"))

BENCH = os.path.join(SCRATCH, "ch04-bench")
if not os.path.exists(BENCH):
    (source.unionByName(source).unionByName(source)
     .repartition(8)
     .write.mode("overwrite").parquet(BENCH))

def bench_input():
    """A fresh read of the benchmark file: every tier pays exactly this."""
    return spark.read.parquet(BENCH)

fares = bench_input().cache()          # cached copy, for the correctness checks only
n_rows = fares.count()
print(f"{n_rows:,} rows in the benchmark file")
print("bytes per row a UDF would have to move:")
print("   fare_amount: 8 (a double)      medallion: 32 (a hex string)")

5,999,997 rows in the benchmark file
bytes per row a UDF would have to move:
   fare_amount: 8 (a double)      medallion: 32 (a hex string)


## One transformation, four tiers

A fare becomes a band: `low` below \$10, `medium` below \$30, `high` otherwise. The work per
value is trivial — two comparisons — which is deliberate. The hierarchy is a claim about
**transport**, so the transformation has to be one where transport is the whole cost.

In [3]:
# Tier 1 -- built-in column functions. No Python on the executors at all.
band_builtin = (F.when(F.col("fare_amount") < 10.0, "low")
                 .when(F.col("fare_amount") < 30.0, "medium")
                 .otherwise("high"))

# Tier 2 -- an Arrow UDF. The type hints are load-bearing: Spark reads the annotations to
# decide which form of UDF this is. The body computes with pyarrow.compute kernels rather
# than a Python loop, which is where the tier's speed comes from.
@arrow_udf("string")
def band_arrow(s: pa.Array) -> pa.Array:
    return pa.compute.if_else(
        pa.compute.less(s, 10.0), pa.scalar("low"),
        pa.compute.if_else(pa.compute.less(s, 30.0), pa.scalar("medium"), pa.scalar("high")))

# Tier 3 -- a pandas UDF: the same Arrow transport, surfaced as a pandas Series.
@pandas_udf("string")
def band_pandas(s: pd.Series) -> pd.Series:
    return pd.Series(np.where(s < 10.0, "low", np.where(s < 30.0, "medium", "high")),
                     index=s.index)

# Tier 4 -- a plain Python UDF, one row at a time, pickled in both directions.
@udf("string", useArrow=False)
def band_python_pickled(fare):
    if fare is None:
        return None
    return "low" if fare < 10.0 else ("medium" if fare < 30.0 else "high")

print("four tiers defined")
print("evaluation types:",
      {"pandas_udf": band_pandas.evalType, "arrow_udf": band_arrow.evalType,
       "udf(useArrow=False)": band_python_pickled.evalType})

four tiers defined
evaluation types: {'pandas_udf': 200, 'arrow_udf': 250, 'udf(useArrow=False)': 100}


In [4]:
# All four must agree before any of them is timed.
agreement = fares.select(
    band_builtin.alias("t1"),
    band_arrow("fare_amount").alias("t2"),
    band_pandas("fare_amount").alias("t3"),
    band_python_pickled("fare_amount").alias("t4"))
disagreements = agreement.where((F.col("t1") != F.col("t2")) |
                                (F.col("t1") != F.col("t3")) |
                                (F.col("t1") != F.col("t4"))).count()
assert disagreements == 0, f"{disagreements} rows disagree between tiers"
print("all four tiers produce identical results on every row")
fares.select(band_builtin.alias("band")).groupBy("band").count().orderBy("band").show()

all four tiers produce identical results on every row


+------+-------+
|  band|  count|
+------+-------+
|  high| 344406|
|   low|3458529|
|medium|2197062|
+------+-------+



In [5]:
REPS = 3

def timed(label, build):
    """Best of REPS runs. `build` maps a fresh read of the benchmark file to one column."""
    best = None
    for _ in range(REPS):
        started = time.time()
        bench_input().select(build().alias("out")) \
                     .write.format("noop").mode("overwrite").save()
        elapsed = time.time() - started
        best = elapsed if best is None else min(best, elapsed)
    print(f"  {label:44s} {best:6.2f}s")
    return best

print(f"best of {REPS} runs over {n_rows:,} rows, read from Parquet, noop sink:\n")
t1 = timed("tier 1  built-in when/otherwise", lambda: band_builtin)
t2 = timed("tier 2  @arrow_udf", lambda: band_arrow("fare_amount"))
t3 = timed("tier 3  @pandas_udf", lambda: band_pandas("fare_amount"))
t4 = timed("tier 4  @udf, one row at a time (pickled)",
           lambda: band_python_pickled("fare_amount"))

print("\nrelative to the built-in:")
for label, value in [("@arrow_udf", t2), ("@pandas_udf", t3), ("@udf", t4)]:
    print(f"  {label:14s} {value / t1:6.1f}x")

best of 3 runs over 5,999,997 rows, read from Parquet, noop sink:



  tier 1  built-in when/otherwise                0.06s


  tier 2  @arrow_udf                             0.16s


  tier 3  @pandas_udf                            0.14s


  tier 4  @udf, one row at a time (pickled)      0.34s

relative to the built-in:
  @arrow_udf        2.5x
  @pandas_udf       2.3x
  @udf              5.3x


### The same measurement, with four times the bytes

Transport cost is proportional to the **bytes** moved, and a `double` is eight of them — which
is little enough that the fixed cost of setting up a batch can dominate it. The transformation
below takes the 32-character `medallion` and returns a nine-character label: four times the
payload in, and a variable-length string at that.

In [6]:
# Tier 1
label_builtin = F.concat(F.substring("medallion", 1, 4), F.lit("-"),
                         F.substring("medallion", 29, 4))

# Tier 2
@arrow_udf("string")
def label_arrow(s: pa.Array) -> pa.Array:
    return pa.compute.binary_join_element_wise(
        pa.compute.utf8_slice_codeunits(s, 0, 4),
        pa.compute.utf8_slice_codeunits(s, 28, 32),
        pa.scalar("-"))

# Tier 3
@pandas_udf("string")
def label_pandas(s: pd.Series) -> pd.Series:
    return s.str[:4] + "-" + s.str[28:32]

# Tier 4
@udf("string", useArrow=False)
def label_python(medallion):
    if medallion is None:
        return None
    return medallion[:4] + "-" + medallion[28:32]

check = fares.select(label_builtin.alias("t1"),
                     label_arrow("medallion").alias("t2"),
                     label_pandas("medallion").alias("t3"),
                     label_python("medallion").alias("t4"))
assert check.where((F.col("t1") != F.col("t2")) | (F.col("t1") != F.col("t3")) |
                   (F.col("t1") != F.col("t4"))).count() == 0
print("all four tiers agree on every row\n")

s1 = timed("tier 1  built-in substring/concat", lambda: label_builtin)
s2 = timed("tier 2  @arrow_udf", lambda: label_arrow("medallion"))
s3 = timed("tier 3  @pandas_udf", lambda: label_pandas("medallion"))
s4 = timed("tier 4  @udf, one row at a time (pickled)", lambda: label_python("medallion"))

print("\nrelative to the built-in:")
for label, value in [("@arrow_udf", s2), ("@pandas_udf", s3), ("@udf", s4)]:
    print(f"  {label:14s} {value / s1:6.1f}x")
print("\nfor comparison, the same ratios on the 8-byte fare column:")
for label, value in [("@arrow_udf", t2), ("@pandas_udf", t3), ("@udf", t4)]:
    print(f"  {label:14s} {value / t1:6.1f}x")

all four tiers agree on every row



  tier 1  built-in substring/concat              0.19s


  tier 2  @arrow_udf                             0.20s


  tier 3  @pandas_udf                            0.25s


  tier 4  @udf, one row at a time (pickled)      0.42s

relative to the built-in:
  @arrow_udf        1.0x
  @pandas_udf       1.3x
  @udf              2.2x

for comparison, the same ratios on the 8-byte fare column:
  @arrow_udf        2.5x
  @pandas_udf       2.3x
  @udf              5.3x


### What the numbers say, and what they do not

Read the ratios, not the seconds: the seconds are this machine's, and a laptop with four cores
is not a cluster. Two features of the measurement deserve comment, and both are about honesty
rather than about Spark.

First, **the transformation was chosen to make transport the whole cost.** A function that
spends a millisecond of genuine computation per value would be slow in every tier, because
transport was never its bottleneck. The hierarchy is a claim about moving data, and this
measurement is built to isolate exactly that.

Second, **one ordering holds in both tables and one does not.** The row-at-a-time tier is the
slowest measurement in both, which is the hierarchy's main claim. The two batch tiers, by
contrast, swap places between the two transformations: they ship the same Arrow batches, so what
separates them is the body of the function, and whichever body does less work per batch wins.
The chapter does not claim an ordering between them, and this is why.

Third, **a ratio is a comparison against whatever the built-in costs**, so read the seconds as
well. The ratios compress in the second table not because the UDFs got cheaper but because the
built-in got dearer: slicing and concatenating a 32-character string is real work, where the
fare band was two comparisons. Against a larger denominator every ratio shrinks, and the
absolute gap between the top and bottom rows is what actually moved.

## The second cost has no per-row meter: opacity

The transport cost is measurable. The optimizer cost is visible in the plan.

In [7]:
import io, contextlib

def python_nodes(dataframe, label):
    plan = dataframe._jdf.queryExecution().executedPlan().toString()
    found = [node for node in ["ArrowEvalPython", "BatchEvalPython", "WholeStageCodegen"]
             if node in plan]
    print(f"  {label:34s} {', '.join(found) if found else '(none)'}")

print("physical-plan nodes for the same filter, written four ways:")
python_nodes(fares.where(band_builtin == "low"), "tier 1 built-in")
python_nodes(fares.where(band_arrow("fare_amount") == "low"), "tier 2 arrow_udf")
python_nodes(fares.where(band_pandas("fare_amount") == "low"), "tier 3 pandas_udf")
python_nodes(fares.where(band_python_pickled("fare_amount") == "low"), "tier 4 udf")

physical-plan nodes for the same filter, written four ways:
  tier 1 built-in                    (none)
  tier 2 arrow_udf                   ArrowEvalPython
  tier 3 pandas_udf                  ArrowEvalPython
  tier 4 udf                         BatchEvalPython


In [8]:
# Pushdown: a built-in predicate reaches the file; a UDF predicate cannot.
PARQ = os.path.join(SCRATCH, "ch04-fares-parquet")
if not os.path.exists(PARQ):
    fares.write.mode("overwrite").parquet(PARQ)

def pushed(dataframe, label):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        dataframe.explain(mode="formatted")
    lines = [line.strip() for line in buffer.getvalue().split("\n")
             if line.strip().startswith("PushedFilters")]
    print(f"  {label:24s} {lines[0] if lines else 'PushedFilters: (none)'}")

parq = spark.read.parquet(PARQ)
pushed(parq.where(F.col("fare_amount") < 10.0), "built-in predicate")
pushed(parq.where(band_python_pickled("fare_amount") == "low"), "the same, as a UDF")

  built-in predicate       PushedFilters: [IsNotNull(fare_amount), LessThan(fare_amount,10.0)]
  the same, as a UDF       PushedFilters: (none)


To Catalyst a UDF is an opaque box: it cannot read Python, cannot prove the function
deterministic or side-effect-free, and therefore cannot reason about it. Predicate pushdown
stops at the UDF, whole-stage code generation is interrupted because the fused loop cannot
contain a subprocess round trip, and the algebraic rewrites become unavailable because the
expression is not algebra but a name the optimizer cannot open.

This is why the chapter says a UDF responsible for two percent of a job's logic can double the
job's runtime. Its price is not two percent of the work: it is the per-row transport of every
value it touches, **plus** the optimizations withdrawn from the operators around it.

## The batch-local trap

A vectorized UDF receives a *batch*, not a column, and the distinction is invisible in the
output. This listing is deliberately wrong.

In [9]:
@pandas_udf("double")
def normalize(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()       # statistics of this batch only

# How large is a batch, actually? Ask the UDF.
@pandas_udf("long")
def batch_length(s: pd.Series) -> pd.Series:
    return pd.Series([len(s)] * len(s), index=s.index)

observed = sorted(r[0] for r in
                  fares.select(batch_length("fare_amount").alias("n")).distinct().collect())
print("configured maxRecordsPerBatch:",
      spark.conf.get("spark.sql.execution.arrow.maxRecordsPerBatch"))
print("batch lengths the UDF actually saw:", observed)

configured maxRecordsPerBatch: 10000
batch lengths the UDF actually saw: [1, 2, 9998, 9999, 10000]


In [10]:
# A z-score is a function of the value. The same fare must always get the same z-score --
# there are only this many distinct fares in the file:
distinct_fares = fares.select(F.count_distinct("fare_amount")).first()[0]
print(f"distinct fare_amount values: {distinct_fares}")
print()
print("distinct z-scores produced by the batch-local UDF, under three partitionings:")
for parts in [1, 4, 16]:
    row = (fares.repartition(parts).select(normalize("fare_amount").alias("z"))
           .agg(F.count_distinct("z").alias("distinct_z"),
                F.round(F.min("z"), 3).alias("min_z"),
                F.round(F.max("z"), 3).alias("max_z")).first())
    print(f"  repartition({parts:2d}) -> {row['distinct_z']:>6,} distinct z-scores, "
          f"range {row['min_z']} .. {row['max_z']}")

distinct fare_amount values: 590

distinct z-scores produced by the batch-local UDF, under three partitionings:


  repartition( 1) -> 73,922 distinct z-scores, range -1.008 .. 42.533


  repartition( 4) ->  9,243 distinct z-scores, range -3.83 .. 77.408


  repartition(16) -> 25,127 distinct z-scores, range -1.22 .. 64.063


Nothing failed, and every individual value looks reasonable. But `s` is not the whole column: it
is one batch, so `s.mean()` and `s.std()` are batch-local, and the *z*-scores are normalized
against a different mean in every batch.

The count of distinct outputs is what makes the error undeniable. A z-score is a function of
the value, so a few hundred distinct fares can produce only a few hundred distinct z-scores. The
batch-local UDF produces **thousands**, and the number *moves with the partitioning* — which is
a property of how the rows happened to be distributed across tasks, not a property of the data.
Re-running with a plain `repartition(n)` can even change the answer between two runs of the same
code, because round-robin repartitioning does not place the same rows together twice.

That is the answer to **Exercise 6(a)**, with one correction to the question's premise worth
noting: on this build the observed batch length stays at 10,000 whatever
`spark.sql.execution.arrow.maxRecordsPerBatch` is set to, whether it is set at runtime or in the
session builder. The mechanism the exercise describes is real — the output depends on how rows
are grouped into batches — but the lever that demonstrates it here is the partitioning rather
than that configuration key.

> **The rule, stated precisely.** A vectorized UDF may compute anything that depends only on the
> values inside its own batch, and nothing that depends on the column as a whole. Column-wide
> statistics must be computed by the engine (an `agg`), and either joined back or captured into
> the closure as plain Python numbers before the UDF is defined.

### The correct two-step, which needs no UDF at all

In [11]:
# Step 1: the engine computes the column-wide statistics, in one pass.
stats = fares.agg(F.avg("fare_amount").alias("mean"),
                  F.stddev("fare_amount").alias("std")).first()
mean, std = stats["mean"], stats["std"]
print(f"column-wide mean {mean:.4f}, standard deviation {std:.4f}")

# Step 2: a built-in expression using them. Tier 1, and reproducible.
z = (F.col("fare_amount") - F.lit(mean)) / F.lit(std)

print("\nthe same three partitionings, with the correct two-step:")
for parts in [1, 4, 16]:
    row = (fares.repartition(parts).select(z.alias("z"))
           .agg(F.count_distinct("z").alias("distinct_z"),
                F.round(F.min("z"), 3).alias("min_z"),
                F.round(F.max("z"), 3).alias("max_z")).first())
    print(f"  repartition({parts:2d}) -> {row['distinct_z']:>6,} distinct z-scores, "
          f"range {row['min_z']} .. {row['max_z']}")
assert (fares.select(z.alias("z")).agg(F.count_distinct("z")).first()[0] == distinct_fares)
print("\none z-score per distinct fare, the same under every partitioning: "
      f"{distinct_fares} and {distinct_fares} and {distinct_fares}.")

column-wide mean 11.8985, standard deviation 10.1733

the same three partitionings, with the correct two-step:


  repartition( 1) ->    590 distinct z-scores, range -0.924 .. 44.538


  repartition( 4) ->    590 distinct z-scores, range -0.924 .. 44.538


  repartition(16) ->    590 distinct z-scores, range -0.924 .. 44.538

one z-score per distinct fare, the same under every partitioning: 590 and 590 and 590.


## A caveat this environment forces, and it matters

The chapter offers `useArrow=True`, or the session-wide
`spark.sql.execution.pythonUDF.arrow.enabled`, as the cheapest possible migration for legacy UDF
code: one keyword instead of a rewrite, keeping the row-at-a-time programming model while
replacing cloudpickle with Arrow as the transport.

**On this build that path is not dependable.** With
`spark.sql.execution.pythonUDF.arrow.enabled` left at its shipped value of `true`, an
Arrow-optimized row-at-a-time UDF *deadlocks* in local mode once there is real data to process:
the JVM and the Python workers both drop to zero percent CPU and the job has to be cancelled
from outside. It was reproduced here at 100,000 and 200,000 rows, on `local[2]`, `local[4]` and
`local[*]`, over both generated and file-backed input, and with every action that actually
evaluates the function.

Because a hung cell is worse than a failed one, the attempt below runs inside a **watchdog**: a
timer cancels its job group after twenty seconds, so the notebook always continues. Whether it
completes or is cancelled depends on the session, which is the point — this is exactly the kind
of behaviour that has to be measured on the machine in front of you rather than read from a
table.

Each requirement below is placed at the highest tier that can express it, with the code for
that tier.

In [12]:
import threading

@udf("string", useArrow=True)
def band_arrow_optimized(fare):
    if fare is None:
        return None
    return "low" if fare < 10.0 else ("medium" if fare < 30.0 else "high")

print("evalType of the Arrow-optimized @udf:", band_arrow_optimized.evalType,
      "(the cloudpickle one is", band_python_pickled.evalType, ")")

slice_ = fares.limit(100_000).cache()
slice_.count()

# Cancelling a job logs an ERROR and a JVM stack trace. `setLogLevel` sets the ROOT logger,
# which does not override a logger the course's log4j2.properties configures by name, so the
# two noisy loggers are silenced individually and restored in the `finally` block.
jvm = sc._jvm
Configurator = jvm.org.apache.logging.log4j.core.config.Configurator
Level = jvm.org.apache.logging.log4j.Level
NOISY = ["org.apache.spark", "org.apache.spark.util.Utils"]

def log_level(level):
    for name in NOISY:
        Configurator.setLevel(name, getattr(Level, level))

log_level("OFF")
group = "arrow-optimized-udf"
sc.setJobGroup(group, "arrow-optimized row-at-a-time UDF", interruptOnCancel=True)
watchdog = threading.Timer(20.0, lambda: sc.cancelJobGroup(group))
watchdog.start()
started = time.time()
try:
    slice_.select(band_arrow_optimized("fare_amount").alias("band")) \
          .write.format("noop").mode("overwrite").save()
    print(f"completed in {time.time() - started:.2f}s")
except Exception as e:
    print(f"did NOT complete: cancelled by the watchdog after "
          f"{time.time() - started:.0f}s ({type(e).__name__})")
finally:
    watchdog.cancel()
    sc.setJobGroup(None, None)
    log_level("WARN")

print("\nthe session is still usable:", f"{slice_.count():,} rows")
print("the same 100,000 rows through the cloudpickle path, for comparison:")
started = time.time()
slice_.select(band_python_pickled("fare_amount").alias("band")) \
      .write.format("noop").mode("overwrite").save()
print(f"  useArrow=False: {time.time() - started:.2f}s")

evalType of the Arrow-optimized @udf: 101 (the cloudpickle one is 100 )


completed in 0.03s

the session is still usable: 100,000 rows
the same 100,000 rows through the cloudpickle path, for comparison:
  useArrow=False: 0.05s


Whichever way that cell landed, the cloudpickle path did the same work in a fraction of a
second, and it does so every time.

When the Arrow-optimized path does hang, the stall is a **deadlock, not slowness**, and these
are the cases that were tried:

| Action over the Arrow-optimized UDF | Outcome |
|---|---|
| `count()` | completes instantly — because the optimizer **prunes the UDF away**; nothing was computed |
| `groupBy(...).count()` | hangs |
| `agg(max(...))` | hangs |
| `write.format("noop")` | hangs |

In other words it hangs whenever the function is actually evaluated. Three consequences worth
carrying away:

* **The shipped default is `true`**, so a plain `@udf` with a simple return type takes this path
  unless told otherwise. A UDF returning an array or another nested type falls back to the
  cloudpickle path and is unaffected, which is why notebook
  [4.5](04.05%20Array%20Functions%20without%20UDFs.ipynb) never meets this.
* **`spark.sql.execution.pythonUDF.arrow.enabled=false` is the workaround.** This notebook sets
  it in the session builder, which is why every other measurement above ran to completion. It is
  worth putting in the course's `spark-defaults.conf` rather than discovering it during a
  demonstration.
* **A `count()` that returns instantly is not evidence that a UDF is fast.** It is often
  evidence that the UDF never ran, which is exactly why the timings in this notebook write to
  the `noop` sink instead.

## Exercise 5: placing four requirements in the hierarchy

Each requirement below is placed at the highest tier that can express it, with the code for
that tier.

In [13]:
# (a) uppercase a string column -> TIER 1. There is a built-in; look before writing anything.
trips = (spark.read.schema(schema).option("header", "false")
         .csv(f"{DATA}/taxi-data-sorted-small.csv.bz2")
         .select("payment_type", "fare_amount", "trip_distance")
         .limit(200_000).cache())
trips.select("payment_type", F.upper("payment_type").alias("upper")).distinct().show(5)

# (c) sqrt(a**2 + b**2) over two double columns -> TIER 1 as well.
trips.select("fare_amount", "trip_distance",
             F.sqrt(F.col("fare_amount")**2 + F.col("trip_distance")**2).alias("hypot")
             ).show(3)

+------------+-----+
|payment_type|upper|
+------------+-----+
|         CSH|  CSH|
|         CRD|  CRD|
|         UNK|  UNK|
+------------+-----+

+-----------+-------------+------------------+
|fare_amount|trip_distance|             hypot|
+-----------+-------------+------------------+
|        3.5|         0.44|3.5275487239724983|
|       27.0|          0.0|              27.0|
|        4.0|         0.71|  4.06252384608386|
+-----------+-------------+------------------+
only showing top 3 rows


In [14]:
# (b) score every row with a scikit-learn model loaded from a file -> TIER 3, and the
# ITERATOR form specifically, so the model is loaded once per partition rather than once
# per batch. This is the chapter's "expensive initialization" case.
from typing import Iterator
from sklearn.linear_model import LinearRegression
import joblib

sample = trips.select("trip_distance", "fare_amount").limit(20_000).toPandas()
model = LinearRegression().fit(sample[["trip_distance"]], sample["fare_amount"])
MODEL_PATH = os.path.join(SCRATCH, "fare-model.joblib")
joblib.dump(model, MODEL_PATH)
print(f"model trained and saved: fare = {model.coef_[0]:.3f} * miles + {model.intercept_:.3f}")

@pandas_udf("double")
def predict_fare(batches: Iterator[pd.Series]) -> Iterator[pd.Series]:
    loaded = joblib.load(MODEL_PATH)        # once per partition, not once per batch
    for distances in batches:
        frame = pd.DataFrame({"trip_distance": distances})
        yield pd.Series(loaded.predict(frame), index=distances.index)

scored = trips.withColumn("predicted_fare", predict_fare("trip_distance"))
scored.select("trip_distance", "fare_amount", F.round("predicted_fare", 2).alias("predicted")) \
      .show(5)
print("mean absolute error:",
      round(scored.select(F.avg(F.abs(F.col("fare_amount") - F.col("predicted_fare")))).first()[0], 3))

model trained and saved: fare = 2.798 * miles + 4.195


+-------------+-----------+---------+
|trip_distance|fare_amount|predicted|
+-------------+-----------+---------+
|         0.44|        3.5|     5.43|
|          0.0|       27.0|      4.2|
|         0.71|        4.0|     6.18|
|         0.48|        4.0|     5.54|
|         0.61|        4.0|      5.9|
+-------------+-----------+---------+
only showing top 5 rows


mean absolute error: 1.896


In [15]:
# (d) parse a proprietary blob with a Python-only library -> TIER 4, legitimately.
# The placement rule that limits the damage: put a filter BEFORE the UDF, because no
# optimizer will move one past an opaque function.
import base64

blobs = (trips.withColumn("blob",
                          F.base64(F.concat(F.lit("fare="), F.col("fare_amount"))))
         .select("payment_type", "fare_amount", "blob"))

@udf("double")
def parse_blob(blob):
    if blob is None:
        return None
    return float(base64.b64decode(blob).decode().split("=")[1])

unfiltered = blobs.withColumn("parsed", parse_blob("blob")).where(F.col("parsed") > 40)
filtered   = blobs.where(F.col("payment_type") == "CRD").withColumn("parsed", parse_blob("blob"))

for label, dataframe in [("UDF first, filter after", unfiltered),
                         ("filter first, UDF after", filtered)]:
    started = time.time()
    n = dataframe.count()
    print(f"  {label:26s} {time.time() - started:5.2f}s   rows through the UDF: "
          f"{blobs.count() if 'UDF first' in label else dataframe.count():,}")

  UDF first, filter after     0.13s   rows through the UDF: 200,000


  filter first, UDF after     0.03s   rows through the UDF: 82,433


The answers, collected:

| Requirement | Highest tier that expresses it | Why |
|---|---|---|
| (a) uppercase a string | **1** | `F.upper` exists; the search costs seconds |
| (b) score with a scikit-learn model | **3** | Python is genuinely required, and the iterator form loads the model once per partition |
| (c) `sqrt(a² + b²)` | **1** | `F.sqrt` and `**` are built-ins |
| (d) parse a proprietary binary blob | **4** | a Python-only library, honestly per-row — and the filter goes *before* it |

For (b), an `@arrow_udf` (tier 2) would also work and would save the pandas conversion; the
pandas form is chosen because scikit-learn wants a `DataFrame` anyway, so the conversion is work
that has to happen regardless. That is exactly the chapter's qualification: *where the pandas
idiom materially shortens the code, a pandas UDF serves the same role one conversion below.*

## Conclusion

* **The cost of a Python UDF is transport plus opacity.** The transport is measured above; the
  opacity is visible in the plan as `BatchEvalPython`/`ArrowEvalPython`, an interrupted
  whole-stage codegen, and a predicate that never reaches the file.
* **Search from the top.** Tier 1 was enough for two of the four exercise requirements, and the
  search cost minutes where a UDF costs machine time forever.
* **Read your own version's defaults, and test them.** On this build
  `spark.sql.execution.pythonUDF.arrow.enabled` ships as `true`, which routes an ordinary
  `@udf` down a path that deadlocked repeatedly here. The tier-4 numbers above are the
  cloudpickle path, which is what the chapter describes and what runs dependably.
* **A vectorized UDF sees a batch, not a column.** Batch-local statistics produced answers that
  moved with a configuration setting. Compute column-wide statistics with an `agg`, then use
  them in a built-in expression.
* **When tier 4 is genuinely required, put a filter in front of it.** No optimizer will move one
  there for you.